# Adding Linkers & Transfers

This notebook shows how linkers and transfers are created without the Constraints class.

In [1]:
from cobra2d import Phase, Phases, Linker, Linkage, Transfer, Transfers

First, we create some phases, which are later connected by the linkers and transfers.

In [2]:
phases = Phases()

for name in ["root", "leaf"]:
    for time in range(1, 4):
        phases.add_phase(
            Phase(
                id=f"{name}_{time}",
                light_dark="light",
                timeframe=1,
                volume=1,
            )
        )

In [3]:
print(phases)

+--------+------+--------+-----------+
| Phase  | Name | Volume | Timeframe |
+--------+------+--------+-----------+
| root_1 |      |   1    |     1     |
| root_2 |      |   1    |     1     |
| root_3 |      |   1    |     1     |
| leaf_1 |      |   1    |     1     |
| leaf_2 |      |   1    |     1     |
| leaf_3 |      |   1    |     1     |
+--------+------+--------+-----------+


Now we load a model and look at how many reactions it contains.

In [4]:
from cobra.io import load_model

model = load_model("textbook")

print(len(model.reactions))

95


After applying the phases, the model consists of several time periods and sub_models and contains the following number of reactions:

In [5]:
model = phases.apply_phases(model)

print(len(model.reactions))

New suffix 'root_1' for model added


Iteration 1 for 'root_2' completed


Iteration 2 for 'root_3' completed


Iteration 3 for 'leaf_1' completed


Iteration 4 for 'leaf_2' completed


Iteration 5 for 'leaf_3' completed


Model e_coli_core successfully modified


570


Now we create a linkage object and apply it to our previously obtained model.

In [6]:
linkage = Linkage()
linkage.append(
    Linker(
        metabolite_id="atp_c",
        source="root_1",
        destination="root_2",
    )
)

model = linkage.apply(model, phases)
print(len(model.reactions))

571


The model now contains one additional reaction, the new linker. Let's take a closer look at it.

In [7]:
model.reactions.get_by_id("atp_c_lk_root_[1|2]")

Reaction identifier,atp_c_lk_root_[1|2]
Name,Linker for atp_c from root_1 to root_2
Memory address,0x7f8d03ed8500
Stoichiometry,atp_c_root_1 --> atp_c_root_2 ATP --> ATP
GPR,
Lower bound,0.0
Upper bound,1000.0


Transfers are created in the same way. Here we transfer glucose from the root to the leaf in time period 1.

In [8]:
transfers = Transfers()
transfers.append(
    Transfer(
        metabolite_id="glc__D_e",
        source="root_1",
        destination="leaf_1",
    )
)

model = transfers.apply(model, phases)
print(len(model.reactions))

572


The transfer reaction has been added as well:

In [9]:
model.reactions.get_by_id("glc__D_e_tr_[root|leaf]_1")

Reaction identifier,glc__D_e_tr_[root|leaf]_1
Name,Transfer for glc__D_e from root_1 to leaf_1
Memory address,0x7f8d03ed95e0
Stoichiometry,glc__D_e_root_1 --> glc__D_e_leaf_1 D-Glucose --> D-Glucose
GPR,
Lower bound,0.0
Upper bound,1000.0


All of this can also be done using the Constraints class, which combines these individual functionalities. It is therefore recommended to use it instead of the manual approach shown here. An example can be found in the notebook 'Constraints.ipynb' or in the documentation under 'Time and space resolved modeling'.